# Soccer momentum: what the pipeline found

Every chart reads a file in `results/`; nothing is recomputed. Rerun the notebook
after new results land (`Kernel -> Restart & Run All`) and every chart updates.

**The question.** After a team creates a chance, is its *next* chance more likely
within a few minutes, once scoreline, red cards, minute of the match and team
strength are accounted for? The answer is the branching ratio $\eta_{self}$: the
fraction of a team's chances set off by its own earlier chances. It is only
meaningful next to the **detection floor**: the smallest $\eta$ the method catches
80% of the time on simulated matches.

Sections follow the build order in `CLAUDE.md`, headline first.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RES = ROOT / "results"
TAG = "pooled_dev"

# Validated palette (dataviz reference, first three slots pass CVD all-pairs).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 10,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 2, "lines.markersize": 7, "legend.frameon": False, "legend.labelcolor": INK2,
})

# The joint-fit iteration cap was raised in commit 76364fd. Anything written before
# that ran under the old cap and is superseded once the ARC rerun lands.
FIX_AT = datetime.fromisoformat("2026-10-08T16:45:21+00:00")


def manifest(name):
    p = RES / f"{name}.json"
    return json.loads(p.read_text()) if p.exists() else None


def table(name):
    p = RES / f"{name}.csv"
    return pd.read_csv(p) if p.exists() else None


def floor_line(ax, floor, *, vertical=True, label=True, at=1.0):
    kw = dict(color=INK2, linestyle=(0, (4, 3)), linewidth=1.5, zorder=3)
    (ax.axvline if vertical else ax.axhline)(floor, **kw)
    if label:
        if vertical:
            ax.annotate(f"detection floor {floor:g}", (floor, at), xycoords=("data", "axes fraction"),
                        xytext=(5, -4), textcoords="offset points", ha="left", va="top", color=INK2)
        else:
            ax.annotate(f"detection floor {floor:g}", (1, floor), xycoords=("axes fraction", "data"),
                        xytext=(-4, 4), textcoords="offset points", ha="right", va="bottom", color=INK2)

## 0. Which results are current

Each row is one step's results file. **pre-fix** means it was computed before the
iteration-cap fix (commit `76364fd`) and will be replaced by the ARC rerun of
`arc/pipeline.slurm`. Treat pre-fix numbers as provisional.

In [ ]:
steps = [
    ("3 null model", "null_pooled_dev"), ("estimator comparison", "estimator_comparison_pooled_dev"),
    ("5 recovery (gate)", "recovery_pooled_dev"), ("6 detection floor", "power_pooled_dev"),
    ("tau profile", "tau_profile_pooled_dev"), ("7 rounding bias", "rounding_bias_pooled"),
    ("8 headline fit", "fit_pooled_dev"),
]
rows = []
for step, name in steps:
    m = manifest(name)
    if m is None:
        rows.append({"step": step, "file": name, "written (UTC)": "missing", "status": "not run"})
        continue
    at = datetime.fromisoformat(m["written_at_utc"])
    rows.append({"step": step, "file": name, "written (UTC)": at.strftime("%Y-%m-%d %H:%M"),
                 "status": "current" if at >= FIX_AT else "pre-fix"})
pd.DataFrame(rows).set_index("step")

## 1. The headline: is there momentum?

Three things on one axis (log scale, because the numbers span ten orders of
magnitude):

- **the histogram**: $\eta$ estimates from 500 simulated datasets with *no*
  momentum. This is what "zero" looks like through this method.
- **the blue line**: $\eta_{self}$ estimated on the real 4,053 matches.
- **the dashed line**: the detection floor. Momentum this large or larger would be
  caught at least 80% of the time.

If the blue line sits inside the histogram and far left of the floor, the data
look exactly like a world without momentum, and any momentum that exists is
smaller than the floor.

In [ ]:
fit = manifest(f"fit_{TAG}")
null = table(f"bootstrap_null_{TAG}")
if fit is None or null is None:
    print("step 8 has not been run yet")
else:
    p = fit["payload"]
    hk = p["headline"]["hawkes"]
    eta, floor = hk["eta_self"], p["detection_floor"]
    lo, hi = p["cluster_bootstrap_ci95"]
    draws = null["eta_self"].clip(lower=1e-14)

    fig, ax = plt.subplots(figsize=(9, 4))
    bins = np.logspace(np.log10(min(draws.min(), eta) / 2), np.log10(floor * 3), 50)
    ax.hist(draws, bins=bins, color=BLUE, alpha=0.28, edgecolor=SURFACE, linewidth=1,
            label="simulated, no momentum (500)")
    ax.axvspan(lo, hi, color=BLUE, alpha=0.10, lw=0, label="real data, 95% CI (match bootstrap)")
    ax.axvline(eta, color=BLUE, linewidth=2.5, label=f"real data: η_self = {eta:.1e}")
    floor_line(ax, floor, label=False)
    ax.annotate(f"detection\nfloor {floor:g}", (floor, 0.98), xycoords=("data", "axes fraction"),
                xytext=(-5, 0), textcoords="offset points", ha="right", va="top", color=INK2)
    n_zero = int((null["eta_self"] <= 1e-14).sum())
    if n_zero:
        ax.annotate(f"{n_zero} runs at ≤10⁻¹⁴\n(estimate hit zero)", (draws.min(), n_zero),
                    xytext=(10, -6), textcoords="offset points", va="top", color=INK2, fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel("η_self  (fraction of chances triggered by an earlier chance)")
    ax.set_ylabel("simulated datasets")
    # ax.set_title("Momentum on the real matches looks exactly like no momentum")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=9)
    plt.show()

    pb = p["parametric_bootstrap"]
    summary = {
        "η_self": f"{eta:.2e}",
        "η_cross": f"{hk['eta_cross']:.2e}",
        "τ (minutes)": f"{hk['tau_minutes']:.2f}",
        "bootstrap p-value": f"{pb['p_value']:.3f}",
        "95% CI (match bootstrap)": f"[{lo:.1e}, {hi:.1e}]",
        "detection floor": f"{floor} (interpolated {p['detection_floor_interpolated']:.3f})",
        "unconverged null fits": pb.get("n_unconverged", "not recorded (pre-fix run)"),
        "verdict": p["verdict"],
    }
    display(pd.Series(summary, name="value").to_frame())

## 2. How sensitive is the method? (step 6, the detection floor)

We plant a known amount of momentum into simulated matches built from the fitted
no-momentum model, run the *entire* pipeline, and see whether it is detected.
"Detected" means the estimate exceeds the 95th percentile of the $\eta^\ast = 0$ arm.

- **Left**: power. The floor is the smallest planted value detected at least 80%
  of the time.
- **Right**: what comes back versus what was planted. Points below the diagonal
  mean the method under-estimates; the gap is the bias.

In [ ]:
pw = manifest(f"power_{TAG}")
sweep = table(f"power_sweep_{TAG}")
if pw is None:
    print("step 6 has not been run yet")
else:
    pp = pw["payload"]
    curve = pd.DataFrame(pp["curve"])
    floor = pp["detection_floor"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))

    a1.plot(curve["eta_star"], curve["power"], color=BLUE, marker="o", zorder=4)
    a1.axhline(pp["target_power"], color=MUTED, linewidth=1, zorder=2)
    a1.annotate("80% power", (1, pp["target_power"]), xycoords=("axes fraction", "data"),
                xytext=(-4, -12), textcoords="offset points", ha="right", color=INK2)
    floor_line(a1, floor, at=0.55)
    a1.set_ylim(-0.03, 1.05)
    a1.set_xlabel("planted η_self")
    a1.set_ylabel("share of runs detected")
    a1.set_title("Power")

    lim = curve["eta_star"].max() * 1.08
    a2.plot([0, lim], [0, lim], color=MUTED, linewidth=1, zorder=2)
    a2.annotate("perfect recovery", (lim * 0.62, lim * 0.62), rotation=38, color=MUTED,
                xytext=(0, 5), textcoords="offset points", ha="center")
    if sweep is not None:
        jit = np.random.default_rng(0).uniform(-0.003, 0.003, len(sweep))
        a2.scatter(sweep["eta_star"] + jit, sweep["eta_hat_self"], s=12, color=BLUE, alpha=0.25,
                   linewidths=0, label="each simulated run", zorder=3)
    a2.errorbar(curve["eta_star"], curve["mean_eta_hat"], yerr=curve["sd_eta_hat"], fmt="o",
                color=BLUE, ecolor=BLUE, elinewidth=1.5, capsize=0, label="mean ± sd", zorder=4)
    a2.set_xlim(-0.01, lim)
    a2.set_ylim(-0.01, lim)
    a2.set_xlabel("planted η_self")
    a2.set_ylabel("estimated η_self")
    a2.set_title("Planted vs recovered")
    a2.legend(loc="upper left", fontsize=9)
    fig.tight_layout()
    plt.show()

    cols = ["eta_star", "n_replicates", "power", "mean_eta_hat", "bias"]
    if "frac_converged" in curve:
        cols.append("frac_converged")
    elif sweep is not None:
        curve = curve.merge(sweep.groupby("eta_star")["hawkes_converged"].mean().rename("frac_converged"),
                            on="eta_star")
        cols.append("frac_converged")
    display(curve[cols].round(4).set_index("eta_star"))

## 3. The recovery gate (step 5)

Plant $\eta = 0.30$ and check the pipeline finds it. If this fails, nothing
downstream means anything. The band is the pass tolerance.

In [ ]:
rec = manifest(f"recovery_{TAG}")
det = table(f"recovery_detail_joint_{TAG}")
if rec is None or det is None:
    print("step 5 has not been run yet")
else:
    r = rec["payload"]["results"]["joint"]
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.axvspan(r["eta_planted"] - r["tolerance"], r["eta_planted"] + r["tolerance"],
               color=AQUA, alpha=0.12, lw=0, label=f"pass band ± {r['tolerance']}")
    ax.axvline(r["eta_planted"], color=INK2, linewidth=1.5, label=f"planted {r['eta_planted']}")
    ax.scatter(det["eta_hat_self"], np.zeros(len(det)) + np.linspace(-0.25, 0.25, len(det)),
               color=BLUE, s=40, zorder=3, label="each replicate")
    ax.axvline(r["mean_eta_hat_self"], color=BLUE, linewidth=2,
               label=f"mean {r['mean_eta_hat_self']:.3f}")
    ax.set_yticks([])
    ax.set_ylim(-0.6, 0.6)
    ax.set_xlabel("estimated η_self")
    ax.set_title(f"Recovery gate: {'PASSED' if r['passed'] else 'FAILED'}  (bias {r['bias']:+.3f})")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    plt.show()

## 4. The no-momentum model (step 3)

This is the model that has to explain away every *boring* reason chances bunch
together. Effects are shown as **rate multipliers**: 1.0 means no change, 1.25
means 25% more shots per minute.

The scoreline effect is the one `CLAUDE.md` warns about: a team that goes ahead
sits back and shoots less. Left out, it would masquerade as *negative* momentum.

In [ ]:
nm = manifest(f"null_{TAG}")
if nm is None:
    print("step 3 has not been run yet")
else:
    bg = nm["payload"]["background"]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [5, 3, 2]})

    def bars(ax, effects, title, xlabel):
        keys = sorted(effects, key=int)
        vals = [np.exp(effects[k]) for k in keys]
        ends = {int(keys[0]): "≤", int(keys[-1]): "≥"}
        labels = [f"{ends.get(int(k), '')}{int(k):+d}" if int(k) else "0 (ref)" for k in keys]
        colors = [MUTED if int(k) == 0 else BLUE for k in keys]
        ax.bar(labels, [v - 1 for v in vals], bottom=1, color=colors, width=0.62)
        for i, v in enumerate(vals):
            if keys[i] != "0":
                ax.annotate(f"×{v:.2f}", (i, v), xytext=(0, 3 if v >= 1 else -3), textcoords="offset points",
                            ha="center", va="bottom" if v >= 1 else "top", color=INK2, fontsize=9)
        ax.axhline(1, color=INK2, linewidth=1)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.grid(axis="x", visible=False)

    bars(axes[0], bg["g_score"], "Scoreline", "goal difference (own − opponent)")
    axes[0].set_ylabel("shot-rate multiplier")
    bars(axes[1], bg["h_red"], "Red cards", "man advantage")
    lo = min(np.exp(min(bg["g_score"].values())), np.exp(min(bg["h_red"].values())))
    hi = max(np.exp(max(bg["g_score"].values())), np.exp(max(bg["h_red"].values())))
    for ax in axes[:2]:
        ax.set_ylim(lo - 0.12, hi + 0.12)

    h = np.exp(bg["theta_home"])
    axes[2].axis("off")
    axes[2].text(0.05, 0.62, f"×{h:.2f}", fontsize=30, fontweight="bold", color=INK, transform=axes[2].transAxes)
    axes[2].text(0.05, 0.44, "home advantage\nin shot rate", fontsize=10, color=INK2, transform=axes[2].transAxes, va="top")
    fig.tight_layout()
    plt.show()

### Shot rate through the match

The minute-of-match effect is a natural cubic spline. Reconstructing the curve
needs the match lengths, so this cell reads the development slate (logged in
`logs/holdout_access.log` like every other read; it never touches holdout).

In [ ]:
try:
    import sys
    sys.path.insert(0, str(ROOT))
    from src.config import BIN_MINUTES
    from src.ingest import loader
    from src.ingest.schema import match_frame
    from src.models.spline import natural_cubic_basis, natural_cubic_knots

    slate = loader.load_slate("pooled", split="dev", caller="notebook")
    matches = match_frame(slate.events)
    n_bins = int(np.ceil(matches["match_T"].astype(float).max() / BIN_MINUTES))
    centres = (np.arange(n_bins) + 0.5) * BIN_MINUTES
    basis = natural_cubic_basis(centres, natural_cubic_knots(len(bg["f_minute"]), 0.0, centres[-1]))
    f = basis @ np.array(bg["f_minute"])

    # Share of match-minutes at each minute: the curve is only informative where there is data.
    T = matches["match_T"].astype(float).to_numpy()
    coverage = (T[:, None] >= centres[None, :]).mean(axis=0)
    keep = coverage >= 0.02
    rel = np.exp(f - np.average(f[keep], weights=coverage[keep]))

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(centres[keep], rel[keep], color=BLUE)
    ax.axhline(1, color=INK2, linewidth=1)
    ax.axvline(45, color=MUTED, linewidth=1)
    ax.annotate("half-time", (45, 1), xycoords=("data", "axes fraction"), xytext=(4, -4),
                textcoords="offset points", va="top", color=INK2)
    ax.set_xlabel("minute of match")
    ax.set_ylabel("shot rate vs match average")
    ax.set_title("Shots per minute through the match (other factors held fixed)")
    plt.show()
except Exception as exc:  # the rest of the notebook needs only results/
    print(f"minute curve skipped: {exc}")

### Does the no-momentum model fit? (time-rescaling check)

If the model is right, the time between shots, measured in "expected shots"
rather than minutes, follows an exponential distribution with mean 1. The plot
compares those rescaled gaps with that ideal; points on the diagonal mean a good fit.

Two readings of the same data, reported together:

- **per team per match** (6,390 small tests): the share that reject at 5% should
  be about 5%.
- **pooled** (all ~93,000 gaps): with this many gaps even a small, systematic
  deviation rejects decisively.

In [ ]:
qq = table(f"qq_poisson_{TAG}")
if nm is None or qq is None:
    print("step 3 has not been run yet")
else:
    tr = nm["payload"]["time_rescaling"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [3, 2]})
    m = max(qq["theoretical"].max(), qq["empirical"].max())
    a1.plot([0, m], [0, m], color=MUTED, linewidth=1)
    a1.plot(qq["theoretical"], qq["empirical"], color=BLUE, marker="o", markersize=3, linewidth=1.5)
    a1.set_xlabel("expected quantile, Exponential(1)")
    a1.set_ylabel("observed quantile, rescaled gaps")
    a1.set_title("Rescaled gaps vs the ideal")

    stats = pd.Series({
        "gaps": f"{tr['n']:,}",
        "mean (ideal 1)": f"{tr['mean_tau']:.3f}",
        "variance (ideal 1)": f"{tr['var_tau']:.3f}",
        "pooled KS statistic": f"{tr['ks_stat']:.4f}",
        "pooled KS p-value": f"{tr['ks_pvalue_subsample']:.1e}",
        "team-matches tested": f"{tr['n_processes']:,}",
        "share rejecting at 5%": f"{tr['frac_reject_05']:.1%}",
        "their p-values uniform? (KS p)": f"{tr['uniformity_ks_pvalue']:.2f}",
    })
    a2.axis("off")
    tb = a2.table(cellText=[[k, v] for k, v in stats.items()], colWidths=[0.68, 0.32], loc="center",
                  cellLoc="left", edges="horizontal")
    tb.auto_set_font_size(False)
    tb.set_fontsize(10)
    tb.scale(1, 1.6)
    for cell in tb.get_celld().values():
        cell.set_edgecolor(GRID)
    a2.set_title("Fit statistics")
    fig.tight_layout()
    plt.show()

## 5. Robustness (step 8 sensitivities)

The headline refitted with one thing changed at a time. Every row should be
judged against the floor: anything left of it is "could not have seen momentum
this small". Values are clipped at $10^{-12}$ so they can be drawn on a log axis.

In [ ]:
if fit is None or "sensitivities" not in fit["payload"]:
    print("step 8 has not been run yet")
else:
    sens = fit["payload"]["sensitivities"]
    names = {
        "headline": "headline (3–8 min box)",
        "kernel_box_1_to_20_min": "kernel box 1–20 min",
        "no_score_state": "scoreline removed",
        "no_red_cards": "red cards removed",
        "penalties_kept": "penalties kept",
        "minute_spline_df_3": "minute curve, 3 knots",
        "minute_spline_df_5": "minute curve, 5 knots",
        "minute_spline_df_9": "minute curve, 9 knots",
        "minute_spline_df_12": "minute curve, 12 knots",
    }
    rows = [(names.get(k, k), v["eta_self"]) for k, v in sens.items() if "eta_self" in v]
    asym = fit["payload"].get("asymmetry_check")
    if asym:
        rows.append(("home/away symmetry freed (η_HH)", asym["eta_HH"]))
        rows.append(("home/away symmetry freed (η_AA)", asym["eta_AA"]))
    df = pd.DataFrame(rows, columns=["variant", "eta_self"])[::-1]
    floor = fit["payload"]["detection_floor"]

    fig, ax = plt.subplots(figsize=(9, 0.42 * len(df) + 1))
    y = np.arange(len(df))
    x = df["eta_self"].clip(lower=1e-12)
    ax.hlines(y, 1e-12, x, color=GRID, linewidth=2, zorder=2)
    colors = [BLUE if v.startswith("headline") else INK2 for v in df["variant"]]
    clipped = (df["eta_self"] < 1e-12).to_numpy()
    ax.scatter(x[~clipped], y[~clipped], color=np.array(colors)[~clipped], s=50, zorder=3)
    ax.scatter(x[clipped], y[clipped], facecolor=SURFACE, edgecolor=INK2, linewidth=1.5, s=50, zorder=3)
    for yi in y[clipped]:
        ax.annotate("< 10⁻¹² (zero)", (1e-12, yi), xytext=(9, 0), textcoords="offset points", va="center",
                    color=INK2, fontsize=9)
    floor_line(ax, floor)
    ax.set_yticks(y, df["variant"])
    ax.set_xscale("log")
    ax.set_xlim(5e-13, floor * 5)
    ax.set_xlabel("η_self (log scale)")
    ax.set_title("Every variant lands far below the detection floor")
    ax.grid(axis="y", visible=False)
    plt.show()

## 6. How long does a chance's boost last? (τ profile)

The kernel timescale τ is fixed at a grid of values and everything else refitted.
**Left**: how much worse the fit is than the best τ on the grid (0 = best).
**Right**: the momentum estimate at each τ. The shaded band is the 3–8 minute box
the headline is fitted in, chosen before the pooled slate was fitted.

A flat left panel means the data cannot tell those timescales apart. A rise at long
τ is excitation on a scale of half a match, which is not the few-minute momentum
the question is about, and is more likely slow within-match swings the
no-momentum model misses.

In [ ]:
tp = table(f"tau_profile_{TAG}")
if tp is None:
    print("tau profile has not been run yet")
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
    for ax in (a1, a2):
        ax.axvspan(3, 8, color=AQUA, alpha=0.12, lw=0)
        ax.annotate("headline box", (5.5, 1), xycoords=("data", "axes fraction"), xytext=(0, -4),
                    textcoords="offset points", ha="center", va="top", color=INK2, fontsize=9)
        ax.set_xscale("log")
        ax.set_xticks(tp["tau_minutes"], [f"{t:g}" for t in tp["tau_minutes"]])
        ax.minorticks_off()
        ax.set_xlabel("τ (minutes)")
    a1.plot(tp["tau_minutes"], tp["delta_loglik_vs_best"], color=BLUE, marker="o")
    a1.set_ylabel("log-likelihood vs best τ")
    a1.set_title("Fit quality")
    a2.plot(tp["tau_minutes"], tp["eta_self"].clip(lower=1e-12), color=BLUE, marker="o")
    a2.set_yscale("log")
    a2.set_ylabel("η_self (log scale)")
    a2.set_title("Momentum estimate")
    fig.tight_layout()
    plt.show()

## 7. Why second-level timing matters (step 7)

The same shots, fitted twice: once with their true second-level times, once
rounded to whole minutes the way Understat records them. Rounding also forces a
coarser rebound rule (same team, same minute), which removes more shots.

On this slate both come back at essentially zero, so the comparison here mainly
shows the cost in events; the power study on Understat (minute clock, floor 0.20)
is where the resolution penalty is stark.

In [ ]:
rb = manifest("rounding_bias_pooled")
if rb is None:
    print("step 7 has not been run yet")
else:
    q = rb["payload"]
    sec, mnt = q["second_resolution"], q["minute_resolution"]
    df = pd.DataFrame({
        "second clock": {"η_self": sec["eta_self"], "τ (min)": sec["tau_minutes"],
                         "shots removed as rebounds": sec["dedup"]["removed_fraction"],
                         "shots fitted": sec["n_events"]},
        "minute clock": {"η_self": mnt["eta_self_mean"], "τ (min)": mnt["tau_mean"],
                         "shots removed as rebounds": mnt["dedup"]["removed_fraction"],
                         "shots fitted": mnt["dedup"]["n_after"]},
    })
    fig, ax = plt.subplots(figsize=(6, 2.6))
    vals = df.loc["shots removed as rebounds"]
    ax.barh(vals.index[::-1], vals.values[::-1] * 100, color=BLUE, height=0.5)
    for i, v in enumerate(vals.values[::-1]):
        ax.annotate(f"{v:.1%}", (v * 100, i), xytext=(4, 0), textcoords="offset points", va="center", color=INK2)
    ax.set_xlabel("% of shots merged as rebounds")
    ax.set_title(f"Rounding to minutes merges {vals['minute clock'] / vals['second clock']:.1f}× as many shots")
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, vals.max() * 130)
    plt.show()
    display(df.T.style.format({"η_self": "{:.1e}", "τ (min)": "{:.2f}",
                               "shots removed as rebounds": "{:.1%}", "shots fitted": "{:,.0f}"}))

## 8. Where do rebounds end? (step 2)

For consecutive shots by the same team, how much more often does a gap of this
length occur than a smooth baseline predicts? A ratio well above 1 at a few seconds
is mechanical follow-up (rebounds, blocked-shot second balls), not momentum. The
dashed line is the threshold where the excess dies; shots closer than that are
merged before fitting.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
for prov, color in (("statsbomb", BLUE), ("wyscout", ORANGE)):
    g = table(f"dedup_gap_profile_{prov}")
    cal = manifest(f"dedup_calibration_{prov}")
    if g is None or cal is None:
        continue
    g = g[g["gap_s"] <= 60]
    ax.plot(g["gap_s"], g["excess_ratio"], color=color, marker="o", markersize=4, linewidth=1.5,
            label=f"{prov} ({cal['payload']['matches']:,} matches)")
    thr = cal["payload"]["threshold_seconds"]
ax.axhline(1, color=INK2, linewidth=1)
ax.axvline(thr, color=INK2, linestyle=(0, (4, 3)), linewidth=1.5)
ax.annotate(f"merge threshold {thr:g} s", (thr, 1), xycoords=("data", "axes fraction"), xytext=(4, -4),
            textcoords="offset points", va="top", color=INK2)
ax.set_yscale("log")
ticks = [0.3, 0.5, 1, 2, 4]
ax.set_yticks(ticks, [f"{t:g}×" for t in ticks])
ax.minorticks_off()
ax.set_xlabel("seconds between a team's consecutive shots")
ax.set_ylabel("observed / baseline")
ax.set_title(f"Mechanical follow-ups end within about {thr:g} seconds")
ax.legend(loc="upper right", fontsize=9)
plt.show()

## 9. Why the joint estimator (appendix)

Three ways to fit the same model, on simulated data with known momentum. The
two-stage method (fit the no-momentum model first, then the kernel) lets the
background absorb the momentum and comes back far too low; that is why the
headline uses the joint fit.

In [ ]:
ec = table(f"estimator_comparison_{TAG}")
if ec is None:
    print("estimator comparison has not been run yet")
else:
    s = ec.groupby(["method", "eta_star"])["eta_hat_self"].mean().unstack(0)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    lim = s.index.max() * 1.1
    ax.plot([0, lim], [0, lim], color=MUTED, linewidth=1)
    ax.annotate("perfect recovery", (lim, lim), xytext=(4, 0), textcoords="offset points", va="center",
                color=MUTED, fontsize=9)
    for method, color, label in (("joint", BLUE, "joint (headline)"), ("em", AQUA, "EM"),
                                 ("two_stage", ORANGE, "two-stage")):
        if method in s:
            ax.plot(s.index, s[method], color=color, marker="o", label=label)
            ax.annotate(label, (s.index[-1], s[method].iloc[-1]), xytext=(6, 0), textcoords="offset points",
                        va="center", color=INK2, fontsize=9)
    ax.set_xlim(-0.01, lim * 1.25)
    ax.set_xlabel("planted η_self")
    ax.set_ylabel("mean estimated η_self")
    ax.set_title("Two-stage fitting swallows the momentum it should find")
    ax.legend(loc="upper left", fontsize=9)
    plt.show()